Control N°1 Optimización


Importacion de librerias y seleccion del Solver

In [1]:
%pip install -q amplpy
from amplpy import AMPL, ampl_notebook



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
ampl = ampl_notebook(modules=['highs'], license_uuid = "default")

AMPL Version 20250901 (MSVC 19.44.35215.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "C:\Users\Manguera\AppData\Roaming\Python\Python313\site-packages\ampl_module_base\bin\ampl.lic".



## **Problema N°1: Optimización de Procesos de Producción**

El objetivo es determinar las horas de operación de dos procesos distintos para minimizar el costo total diario, asegurando que se cumplan las demandas mínimas de tres químicos específicos.

### **Variables de Decisión**
* $x_1$: Horas de operación del **Proceso 1** por día.
* $x_2$: Horas de operación del **Proceso 2** por día.

### **Función Objetivo**
Minimizar el costo total operativo diario ($Z$):
$$\text{Min } Z = 40x_1 + 10x_2$$

### **Restricciones**
Sujeto a:

1. **R1** (Demanda del químico A): $3x_1 + x_2 \geq 40$
2. **R2** (Demanda del químico B): $x_1 + x_2 \geq 15$
3. **R3** (Demanda del químico C): $x_1 \geq 5$
4. **R4** (No negatividad): $x_1, x_2 \geq 0$

In [ ]:
# Definir el modelo AMPL
model ="""
var x1 >= 0;  # Horas del proceso 1
var x2 >= 0;  # Horas del proceso 2

# Función objetivo:
minimize costo: 40*x1 + 10*x2;

# Restricciones
subject to
  demanda_A: 3*x1 + x2 >= 40;    # Demanda del químico A
  demanda_B: x1 + x2 >= 15;      # Demanda del químico B
  demanda_C: x1 >= 5;             # Demanda del químico C
"""

# Resolver el modelo

ampl.reset()
ampl.eval(model)
ampl.option["solver"] = "highs"
ampl.solve()

# Mostrar resultados
print("Estado solución:", ampl.solve_result)
print(f"\nValor óptimo: {ampl.obj['costo'].value():.2f}")
print(f"\nVariables de decisión:")
print(f"x1 (Proceso 1): {ampl.var['x1'].value():.2f} horas")
print(f"x2 (Proceso 2): {ampl.var['x2'].value():.2f} horas")

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 450
0 simplex iterations
0 barrier iterations
Estado solución: solved

Valor óptimo (Z): 450.00

Variables de decisión:
x1 (Proceso 1): 5.00 horas
x2 (Proceso 2): 25.00 horas


## Modelo Matemático: Mezcla y Procesamiento con Subproductos

Este es un problema de Programación Lineal (PL) de mezcla y procesamiento con subproductos (desechos) que deben ser balanceados o gestionados.

### Variables de Decisión
* $M_1$: Libras de materia prima compradas para elaborar el Insumo 1.
* $M_2$: Libras de materia prima compradas para elaborar el Insumo 2.
* $A$: Onzas de Producto A producidas (procesos tipo 1).
* $B$: Onzas de Producto B producidas (procesos tipo 2).
* $C$: Onzas de Producto C fabricadas.
* $D$: Onzas de Producto D fabricadas.
* $R$: Onzas de desecho líquido vertidas al río.

### Función Objetivo
Maximizar la utilidad ($Z$), que es el ingreso por ventas menos los costos totales (costo de compra y procesamiento de materia prima, más costos de los procesos de los productos):
$$\text{Max } Z = 17A + 16B + 7C + 2D - 8M_1 - 10M_2$$

### Restricciones
Sujeto a:
$$ 
\begin{align*}
2M_1 &= 2A + B + 2C && \text{(Balance de Insumo 1)} \\
3M_2 &= A + 2B + 2D && \text{(Balance de Insumo 2)} \\
A + 0.8B &= R + 0.8C + 1.2D && \text{(Balance de Desecho Líquido)} \\
R &\leq 1000 && \text{(Límite Gubernamental del Río)} \\
A &\leq 5000 && \text{(Demanda Máxima Producto A)} \\
B &\leq 5000 && \text{(Demanda Máxima Producto B)} \\
2M_1 + 2M_2 + 2A + 3B + C + D &\leq 6000 && \text{(Disponibilidad de Tiempo Total)} \\
M_1, M_2, A, B, C, D, R &\geq 0 && \text{(No negatividad)}
\end{align*}
$$